# Extract operculum ROIs

For every head image of `data/<GROUP>`, masks the predicted operculum polygon (`labels/operculum`, class 0), crops it to its bounding box, resizes it to 1080 x 1080 and saves it to `images/operculum_images_extracted/`. Set `GROUP` to `control`, `stressed` or `misc`.

In [1]:
import os
import numpy as np
from PIL import Image, ImageDraw

In [3]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "fish_ids.csv").is_file())
GROUP = "stressed"
DATA = ROOT / "data" / GROUP

imgs_dir = DATA / "images" / "head_images"
lbls_dir = DATA / "labels" / "operculum"
output_dir = DATA / "images" / "operculum_images_extracted"

os.makedirs(output_dir, exist_ok=True)


def load_paths(directory):
    """
    Load all files from a directory into a dictionary.

    Dictionary format:
        {
            filename_without_extension: full_file_path
        }
    """
    paths = {}

    for file_name in sorted(os.listdir(directory)):
        key = os.path.splitext(file_name)[0]
        paths[key] = os.path.join(directory, file_name)

    return paths


img_paths = load_paths(imgs_dir)
lbl_paths = load_paths(lbls_dir)


def load_bbox_details(label_paths):
    """
    Load oriented bounding box / polygon annotations.

    Expected label format per line:
        class_id x1 y1 x2 y2 ... confidence

    This function removes the final value, which is assumed
    to be the confidence score.

    Returns:
        dict:
            {
                filename: [
                    [class_id, x1, y1, x2, y2, ...],
                    ...
                ]
            }
    """
    labels = {}

    for key, path in label_paths.items():
        with open(path, "r") as file:
            lines = file.readlines()

        labels[key] = [
            line.split()[:-1] for line in lines
        ]

    return labels


lbls_det = load_bbox_details(lbl_paths)


def sorting_custom(file_key):
    """
    Sort files based on fish ID and side.

    Expected filename structure:
        something_<fish_id>_..._<side>_something

    Example:
        fish_10_S_day1_id-51_L_h
    """
    try:
        parts = file_key.split("_")

        fish_id = int(parts[1])
        side = parts[-2]

        return fish_id, side

    except Exception as error:
        print(f"Could not sort file key: {file_key}. Error: {error}")
        return float("inf"), ""


sorted_fish_data = dict(
    sorted(lbls_det.items(), key=lambda x: sorting_custom(x[0]))
)


def polygon_to_bounding_box(polygon):
    """
    Convert polygon coordinates to a bounding box.

    Args:
        polygon (np.ndarray): Array of shape (N, 2), where each row is (x, y).

    Returns:
        tuple:
            (min_x, min_y, max_x, max_y)
    """
    x_values = polygon[:, 0]
    y_values = polygon[:, 1]

    min_x = int(np.min(x_values))
    max_x = int(np.max(x_values))
    min_y = int(np.min(y_values))
    max_y = int(np.max(y_values))

    return min_x, min_y, max_x, max_y


def extract_operculum_rois(
    sorted_labels,
    image_paths,
    output_directory,
    output_size=(1080, 1080)
):
    """
    Extract operculum regions from fish images using polygon annotations.

    For each image:
    - Load the original image.
    - Find annotations with class label 0.
    - Convert normalized polygon coordinates to pixel coordinates.
    - Create a binary mask for the polygon.
    - Keep only the polygon region and set the background to black.
    - Crop the masked region using its bounding box.
    - Resize the crop to 1080 x 1080.
    - Save the cropped ROI.

    Args:
        sorted_labels (dict): Sorted annotation dictionary.
        image_paths (dict): Dictionary of image paths.
        output_directory (str): Directory where extracted ROIs are saved.
        output_size (tuple): Final saved image size.
    """

    for filename, annotations in sorted_labels.items():

        if filename not in image_paths:
            print(f"No matching image found for: {filename}")
            continue

        img = Image.open(image_paths[filename]).convert("RGB")
        img_width, img_height = img.size

        for annotation_index, annotation in enumerate(annotations):
            values = [float(x) for x in annotation]

            class_label = int(values[0])
            vertices = values[1:]

            if class_label != 0:
                continue

            pixel_vertices = [
                int(vertices[i] * img_width) if i % 2 == 0
                else int(vertices[i] * img_height)
                for i in range(len(vertices))
            ]

            points = np.array(pixel_vertices).reshape(-1, 2)

            bbox = polygon_to_bounding_box(points)

            mask = Image.new("L", img.size, 0)

            ImageDraw.Draw(mask).polygon(
                [tuple(point) for point in points],
                outline=255,
                fill=255
            )

            black_background = Image.new("RGB", img.size, (0, 0, 0))

            roi = Image.composite(
                img,
                black_background,
                mask
            )

            cropped_roi = roi.crop(bbox)

            cropped_roi = cropped_roi.resize(output_size)

            output_path = os.path.join(
                output_directory,
                f"{filename}.jpg"
            )

            cropped_roi.save(
                output_path,
                format="jpeg",
                subsampling=0,
                quality=100
            )

            print(f"Saved: {output_path}")


extract_operculum_rois(
    sorted_labels=sorted_fish_data,
    image_paths=img_paths,
    output_directory=output_dir,
    output_size=(1080, 1080)
)

Saved: operculums_test/fish_10_S_day1_id-51_L_h.jpg
Saved: operculums_test/fish_10_S_day2_id-131_L_h.jpg
Saved: operculums_test/fish_10_S_day1_id-51_R_h.jpg
Saved: operculums_test/fish_10_S_day2_id-131_R_h.jpg
Saved: operculums_test/fish_11_S_day1_id-63_L_h.jpg
Saved: operculums_test/fish_11_S_day2_id-132_L_h.jpg
Saved: operculums_test/fish_11_S_day1_id-63_R_h.jpg
Saved: operculums_test/fish_11_S_day2_id-132_R_h.jpg
Saved: operculums_test/fish_12_S_day1_id-82_L_h.jpg
Saved: operculums_test/fish_12_S_day2_id-134_L_h.jpg
Saved: operculums_test/fish_12_S_day1_id-82_R_h.jpg
Saved: operculums_test/fish_12_S_day2_id-134_R_h.jpg
Saved: operculums_test/fish_13_S_day1_id-99_L_h.jpg
Saved: operculums_test/fish_13_S_day2_id-135_L_h.jpg
Saved: operculums_test/fish_13_S_day1_id-99_R_h.jpg
Saved: operculums_test/fish_13_S_day2_id-135_R_h.jpg
Saved: operculums_test/fish_14_S_day1_id-61_L_h.jpg
Saved: operculums_test/fish_14_S_day2_id-138_L_h.jpg
Saved: operculums_test/fish_14_S_day1_id-61_R_h.jpg
Sav